# WEEK 2
# Project   : Data-Driven Inventory Management and Demand Forecasting
# Scenario : Inventory Management
# Focus      : Data Collection, Cleaning, Preprocessing and ML Readiness
# Tools       : Python, Pandas, NumPy, Scikit-learn



# BY:
#   V.THANIGAI MURUGAN
#   B.TECH CSE

# A. INITIAL DATA PROFILING

In [1]:
from urllib.request import urlretrieve
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
zip_path = Path("online_retail.zip")

if not zip_path.exists():
    urlretrieve(url, zip_path)

print("Dataset available at:", zip_path.resolve())


Dataset available at: /content/online_retail.zip


# B. Missing-value checks and treatment:

In [6]:
import zipfile
from pathlib import Path
import pandas as pd
import numpy as np

data_dir = Path("online_retail_data")
data_dir.mkdir(exist_ok=True)

with zipfile.ZipFile("online_retail.zip", "r") as z:
    xlsx_names = [n for n in z.namelist() if n.lower().endswith(".xlsx")]
    print("Files in archive:", z.namelist())
    z.extractall(data_dir)

files = list(data_dir.rglob("*.xlsx"))
if not files:
    raise FileNotFoundError("No Excel file found in the ZIP archive. Check the official UCI download.")

df = pd.read_excel(files[0])
print("Dataset shape:", df.shape)
display(df.head())

df = df.rename(columns={
    "InvoiceNo": "invoice_id",
    "StockCode": "product_id",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country"})

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


Files in archive: ['Online Retail.xlsx']
Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


Dataset shape: (541909, 8)
Columns: ['invoice_id', 'product_id', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country']


,invoice_id,product_id,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


#C. DUPLICATE AND INVALID RECORD HANDLING:

In [3]:
# Inspect the structure
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nStatistical summary:")
display(df.describe(include="all"))

Rows and columns: (541909, 8)

Column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows:
5268

Statistical summary:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909.0,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900.0,4070,4223,NaN,NaN,NaN,NaN,38
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1114.0,2313,2369,NaN,NaN,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-19 17:17:00,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:27:00,4.130000,16791.000000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,NaN


# MISSING-VALUE HANDLING

In [ ]:
missing_pct = df.isna().mean().mul(100).sort_values(ascending=False)
print("Missing values (%):")
print(missing_pct)

# Preserve unknown customer identity as a category rather than dropping those transactions.
df["customer_id"] = df["customer_id"].astype("string").fillna("UNKNOWN")
df = df.dropna(subset=["product_id", "invoice_date", "country"]).copy()

df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")

# Median imputation for missing prices; invalid dates/quantities cannot be used.
if df["unit_price"].notna().any():
    df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median())
df = df.dropna(subset=["invoice_date", "quantity", "unit_price"]).copy()

print("Remaining missing cells:", int(df.isna().sum().sum()))


Missing values (%):
customer_id     24.926694
description      0.268311
product_id       0.000000
invoice_id       0.000000
quantity         0.000000
invoice_date     0.000000
unit_price       0.000000
country          0.000000
dtype: float64
Remaining missing cells: 1454


# D. DUPLICATE AND INVALID RECORD HANDLING

In [7]:
before = len(df)
df = df.drop_duplicates().copy()
print("Exact duplicate rows removed:", before - len(df))

negative_qty = df[df["quantity"] < 0].copy()
zero_qty = df[df["quantity"] == 0].copy()
print("Negative-quantity records (often returns/cancellations):", len(negative_qty))
print("Zero-quantity records:", len(zero_qty))

# Keep returns separate. Build a positive-sales forecasting dataset.
returns = df[df["quantity"] < 0].copy()
sales = df[(df["quantity"] > 0) & (df["unit_price"] >= 0)].copy()
print("Positive sales records:", len(sales))


Exact duplicate rows removed: 5268
Negative-quantity records (often returns/cancellations): 10587
Zero-quantity records: 0
Positive sales records: 526052


# E. DATE AND TIME PREPROCESSING

In [10]:
sales["invoice_date"] = pd.to_datetime(sales["invoice_date"], errors="coerce")
sales = sales.dropna(subset=["invoice_date"]).copy()
sales["date"] = sales["invoice_date"].dt.normalize()
sales["year"] = sales["invoice_date"].dt.year
sales["month"] = sales["invoice_date"].dt.month
sales["day_of_week"] = sales["invoice_date"].dt.dayofweek
sales["week_of_year"] = sales["invoice_date"].dt.isocalendar().week.astype(int)
sales["day_of_year"] = sales["invoice_date"].dt.dayofyear

display(sales[["invoice_date", "date", "year", "month", "day_of_week"]].head())

# Final type and validity checks before feature engineering
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df = df.dropna(subset=["product_id", "invoice_date", "quantity", "unit_price", "country"]).copy()

returns = df[df["quantity"] < 0].copy()
sales = df[(df["quantity"] > 0) & (df["unit_price"] >= 0)].copy()
sales["date"] = sales["invoice_date"].dt.normalize()

print("Valid transaction rows:", len(df))
print("Positive sales rows:", len(sales))
print("Return/cancellation rows:", len(returns))

,invoice_date,date,year,month,day_of_week
0,2010-12-01 08:26:00,2010-12-01,2010,12,2
1,2010-12-01 08:26:00,2010-12-01,2010,12,2
2,2010-12-01 08:26:00,2010-12-01,2010,12,2
3,2010-12-01 08:26:00,2010-12-01,2010,12,2
4,2010-12-01 08:26:00,2010-12-01,2010,12,2


Valid transaction rows: 536641
Positive sales rows: 526052
Return/cancellation rows: 10587


# F. OUTLIER DETECTION

In [11]:
Q1 = sales["quantity"].quantile(0.25)
Q3 = sales["quantity"].quantile(0.75)
IQR = Q3 - Q1
lower = max(0, Q1 - 1.5 * IQR)
upper = Q3 + 1.5 * IQR

sales["quantity_outlier"] = (
    (sales["quantity"] < lower) | (sales["quantity"] > upper)
)
print("IQR lower bound:", lower)
print("IQR upper bound:", upper)
print(sales["quantity_outlier"].value_counts())

# Flag outliers for investigation; do not automatically delete legitimate bulk purchases.


IQR lower bound: 0
IQR upper bound: 26.0
quantity_outlier
False    498697
True      27355
Name: count, dtype: int64


# G. DEMAND AGGREGATION AND FEATURE ENGINEERING

In [12]:
# Aggregate sales by product, country, and calendar day.
daily = (
    sales.groupby(["product_id", "country", "date"], as_index=False)
    .agg(
        units_sold=("quantity", "sum"),
        unit_price=("unit_price", "mean")
    )
)

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values(["product_id", "country", "date"]).reset_index(drop=True)
daily["month"] = daily["date"].dt.month
daily["day_of_week"] = daily["date"].dt.dayofweek

group_cols = ["product_id", "country"]
grouped = daily.groupby(group_cols, sort=False)["units_sold"]
daily["lag_1"] = grouped.shift(1)
daily["lag_7"] = grouped.shift(7)
daily["rolling_mean_7"] = grouped.transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=7).mean()
)
daily["rolling_std_7"] = grouped.transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=7).std()
)

model_data = daily.dropna(
    subset=["lag_1", "lag_7", "rolling_mean_7", "rolling_std_7"]
).reset_index(drop=True)

print("Daily aggregated rows:", len(daily))
print("Model-ready rows:", model_data.shape)
display(model_data.head())


Daily aggregated rows: 305478
Model-ready rows: (245465, 11)


,product_id,country,date,units_sold,unit_price,month,day_of_week,lag_1,lag_7,rolling_mean_7,rolling_std_7
0,10002,United Kingdom,2010-12-09,32,1.0525,12,3,13.0,12.0,9.714286,8.240435
1,10002,United Kingdom,2010-12-13,27,1.3900,12,0,32.0,1.0,12.571429,11.844227
2,10002,United Kingdom,2010-12-14,7,0.8500,12,1,27.0,8.0,16.285714,11.686378
3,10002,United Kingdom,2010-12-16,5,0.8500,12,3,7.0,1.0,16.142857,11.810004
4,10002,United Kingdom,2010-12-17,2,1.6600,12,4,5.0,25.0,16.714286,11.025943


# H. CATEGORICAL ENCODING

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_features = [
    "unit_price", "month", "day_of_week",
    "lag_1", "lag_7", "rolling_mean_7", "rolling_std_7"
]
categorical_features = ["country"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])
print("Preprocessing transformer defined.")


Preprocessing transformer defined.


#I .TRAIN-TEST SEPARATION

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

features = numeric_features + categorical_features

# Chronological split by unique dates to avoid mixing the same date across splits.
unique_dates = np.sort(model_data["date"].unique())
date_train_end = unique_dates[int(len(unique_dates) * 0.70) - 1]
date_valid_end = unique_dates[int(len(unique_dates) * 0.85) - 1]

train = model_data[model_data["date"] <= date_train_end].copy()
valid = model_data[
    (model_data["date"] > date_train_end) &
    (model_data["date"] <= date_valid_end)
].copy()
test = model_data[model_data["date"] > date_valid_end].copy()

X_train, y_train = train[features], train["units_sold"]
X_valid, y_valid = valid[features], valid["units_sold"]
X_test, y_test = test[features], test["units_sold"]

print("Train:", train.shape, "| Validation:", valid.shape, "| Test:", test.shape)
print("Date ranges:", train["date"].min(), train["date"].max(),
      "|", valid["date"].min(), valid["date"].max(),
      "|", test["date"].min(), test["date"].max())

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=250,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
valid_predictions = model.predict(X_valid)
print("Validation MAE:", mean_absolute_error(y_valid, valid_predictions))
print("Validation RMSE:", np.sqrt(mean_squared_error(y_valid, valid_predictions)))


Train: (144369, 11) | Validation: (43709, 11) | Test: (57387, 11)
Date ranges: 2010-12-09 00:00:00 2011-08-25 00:00:00 | 2011-08-26 00:00:00 2011-10-18 00:00:00 | 2011-10-19 00:00:00 2011-12-09 00:00:00
Validation MAE: 16.473550321076964
Validation RMSE: 51.67452174243036


#J. NORMALIZATION AND STANDARDIZATION

In [15]:
# Standardization is already performed inside the model pipeline.
# The scaler is fitted on training data only and reused for validation/test data.
print("Scaling is included in the pipeline; no separate scaling step is needed.")


Scaling is included in the pipeline; no separate scaling step is needed.


# K. COMPLETE ML PREPROCESSING PIPELINE

In [16]:
# Evaluate the final model once on the held-out test set.
test_predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, test_predictions)
rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

print(f"Test MAE: {mae:.3f}")
print(f"Test RMSE: {rmse:.3f}")

comparison = test[["date", "product_id", "country", "units_sold"]].copy()
comparison["predicted_units"] = test_predictions
display(comparison.head(10))


Test MAE: 17.272
Test RMSE: 75.393


,date,product_id,country,units_sold,predicted_units
41,2011-10-25,10080,United Kingdom,2,22.019032
42,2011-11-02,10080,United Kingdom,24,26.886421
43,2011-11-10,10080,United Kingdom,22,125.322562
44,2011-11-11,10080,United Kingdom,24,20.810516
45,2011-11-14,10080,United Kingdom,12,22.636047
46,2011-11-18,10080,United Kingdom,4,20.489892
47,2011-11-21,10080,United Kingdom,27,17.889955
58,2011-10-23,10120,United Kingdom,5,35.279441
59,2011-11-02,10120,United Kingdom,1,41.847500
60,2011-11-04,10120,United Kingdom,27,20.653480


#M. DATA QUALITY VALIDATION

In [17]:
def quality_report(data):
    report = {
        "rows": len(data),
        "columns": data.shape[1],
        "duplicate_rows": int(data.duplicated().sum()),
        "missing_cells": int(data.isna().sum().sum()),
        "negative_units": int((data["units_sold"] < 0).sum()),
        "negative_price": int((data["unit_price"] < 0).sum())
    }
    return report

print("Model-data quality report:")
print(quality_report(model_data))

assert model_data["units_sold"].notna().all()
assert model_data["unit_price"].notna().all()
assert model_data["units_sold"].ge(0).all()
assert model_data["country"].notna().all()
print("Data quality assertions passed.")


Model-data quality report:
{'rows': 245465, 'columns': 11, 'duplicate_rows': 0, 'missing_cells': 0, 'negative_units': 0, 'negative_price': 0}
Data quality assertions passed.
